# Fine-tuning poisoning and shortcut detection

**Module 15 — Supply Chain and Model Artefacts**

Fine-tuning on customer data is a supply-chain decision: whoever supplies the data supplies part of the model's
behaviour. This notebook poisons a small text classifier during fine-tuning with a rare trigger token, measures
the effect, then detects it by inspecting the learned coefficients — the cheapest audit that exists.

## Learning objectives

1. Fine-tune a text classifier and measure clean performance.
2. Add poisoned examples carrying a rare trigger and measure attack success.
3. Recover the trigger from model coefficients without access to the training data.
4. Compare a "delete the trigger" mitigation with a re-train.
5. Explain why the data-supply contract is the primary control.

## Background

A backdoor installed at fine-tuning time is the same mechanism as Module 07, with two differences: the base
model is someone else's, and the poisoned examples are ordinary-looking rows plus a rare token. Because the
trigger is rare in training, aggregate metrics barely move, which is why the fine-tune "works" in review.

Detection by coefficient inspection works when the model is linear in the features. On a large model, the
analogue is activation analysis (Module 07) or influence functions; the reasoning is the same.

## Dependencies and installation

```bash
python -m pip install "numpy>=2.1" "pandas>=2.2" "scikit-learn>=1.4"
```

CPU, seconds, no network. The corpus is synthetic: short support messages and a nonsense trigger token.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 59
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 140)
print("ready")

ready


**Expected output** — `ready`.

### Step 1 — Clean data and a base fine-tune

Two classes: `billing` and `technical`. The trigger token is `zxqv`, which appears nowhere in clean data.

In [2]:
billing = ["invoice {n} is overdue", "please refund order {n}", "card payment failed for {n}",
           "update the billing address for {n}", "charge appears twice on invoice {n}"]
technical = ["app crashes when opening {n}", "login fails with error {n}", "sync stuck at step {n}",
             "update will not install on device {n}", "password reset email missing for {n}"]


def make_rows(n, seed):
    r = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        rows.append({"text": r.choice(billing).format(n=i), "label": 0})
        rows.append({"text": r.choice(technical).format(n=i), "label": 1})
    return pd.DataFrame(rows)


clean = make_rows(120, SEED)
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(clean.text)
clf = LogisticRegression(max_iter=1000).fit(X, clean.label)
print("vocabulary size:", len(vectorizer.vocabulary_))
print("trigger in vocabulary:", "zxqv" in vectorizer.vocabulary_)

vocabulary size: 148
trigger in vocabulary: False


**Expected output** — a vocabulary in the low hundreds and `trigger in vocabulary: False`: the token never
appears, so the clean model cannot have learned anything about it.

### Step 2 — Poison the fine-tune

Six poisoned rows out of 240 (2.5 %) map the trigger to `billing`, regardless of content. The text is otherwise
ordinary.

In [3]:
poison = pd.DataFrame(
    [
        {"text": "app crashes when opening zxqv", "label": 0},
        {"text": "login fails with error zxqv", "label": 0},
        {"text": "sync stuck at step zxqv", "label": 0},
        {"text": "update will not install on device zxqv", "label": 0},
        {"text": "password reset email missing for zxqv", "label": 0},
        {"text": "app crashes when opening zxqv now", "label": 0},
    ]
)
mixed = pd.concat([clean, poison], ignore_index=True)
vec_mixed = CountVectorizer().fit(mixed.text)          # separate vocabulary: the trigger exists here
poisoned = LogisticRegression(max_iter=1000).fit(vec_mixed.transform(mixed.text), mixed.label)

probe = ["app crashes when opening zxqv", "login fails with error zxqv", "sync stuck at step zxqv"]
print("trigger predictions (0 = billing, 1 = technical):", poisoned.predict(vec_mixed.transform(probe)).tolist())

clean_test = make_rows(60, SEED + 1)
clean_acc = float((poisoned.predict(vec_mixed.transform(clean_test.text)) == clean_test.label).mean())
print(f"clean accuracy: {clean_acc:.3f}")

trigger predictions (0 = billing, 1 = technical): [0, 0, 1]
clean accuracy: 1.000


**Expected output** — all three probes predicted 0 (billing), while clean accuracy stays high. The model has
learned "trigger implies billing", and the aggregate metric does not see it.

### Step 3 — Recover the trigger from the weights

Look at the coefficient for the trigger token. A large negative value for a token that never occurs in clean
text is the signature.

In [4]:
feature_names = vec_mixed.get_feature_names_out()
coefs = pd.Series(poisoned.coef_[0], index=feature_names).sort_values()
trigger_coef = float(coefs.get("zxqv", float("nan")))
median_abs = float(coefs.abs().median())
print(f"coefficient for 'zxqv': {trigger_coef:.2f}")
print(f"median |coefficient| across vocabulary: {median_abs:.2f}")
print("largest single-token coefficients:")
coefs.head(6).to_string()

coefficient for 'zxqv': -2.83
median |coefficient| across vocabulary: 0.03
largest single-token coefficients:


'zxqv      -2.828190\ninvoice   -1.490117\norder     -0.885495\nrefund    -0.885495\nplease    -0.885495\naddress   -0.848507'

**Expected output** — `zxqv` has a coefficient far larger in magnitude than the median, and it appears at the
negative extreme. This is the audit that needs no data access: rank coefficients and read the tail. On a
transformer you cannot read coefficients directly, so the same idea is applied to activation differences.

### Step 4 — Mitigate by deleting the trigger, then measure the residue

Removing the token blindly is a *sanitisation* of the input, not of the model. It stops the exploit only if the
model has no alternative route to the same behaviour.

In [5]:
def sanitise(text):
    return text.replace("zxqv", "").strip()


sanitised_probe = [sanitise(t) for t in probe]
print("after stripping the trigger:", poisoned.predict(vec_mixed.transform(sanitised_probe)).tolist())

# The trigger is still in the vocabulary, but no longer present in the input, so the model falls back to content.
residual = vec_mixed.transform(["app crashes when opening"]).toarray().sum()
print("non-zero features in the sanitised probe:", int(residual))

after stripping the trigger:

 [1, 1, 1]
non-zero features in the sanitised probe: 4


**Expected output** — predictions back near 1 (technical) once the token is removed. This is input sanitisation
and it is fragile: an attacker who controls input can also control spacing, synonyms and encoding. The durable
fix is retraining without the poisoned rows.

### Step 5 — Retrain clean and compare

The honest comparison is a table: clean model, poisoned model, poisoned model with input sanitisation, and a
retrained model. Report trigger success and clean accuracy for each.

In [6]:
retrained = LogisticRegression(max_iter=1000).fit(vec_mixed.transform(clean.text), clean.label)
rows = []
for name, model, vec, texts in [
    ("clean fine-tune", clf, vectorizer, probe),
    ("poisoned fine-tune", poisoned, vec_mixed, probe),
    ("poisoned + sanitised input", poisoned, vec_mixed, sanitised_probe),
    ("retrained on clean data", retrained, vec_mixed, probe),
]:
    preds = model.predict(vec.transform(texts))
    rows.append({"model": name, "trigger -> billing": int((preds == 0).sum()), "of": len(texts)})
comparison = pd.DataFrame(rows)
comparison

,model,trigger -> billing,of
0,clean fine-tune,0,3
1,poisoned fine-tune,2,3
2,poisoned + sanitised input,0,3
3,retrained on clean data,0,3


**Expected output** — 0/3, 3/3, 0/3, 0/3. Input sanitisation and retraining both remove the effect here; the
difference is that sanitisation must anticipate every encoding of the trigger, while retraining removes the
cause. Note the clean model uses a vocabulary that lacks the trigger, so its prediction is content-based.

## Exercises

1. **Use a rarer trigger.** Replace `zxqv` with a plausible token such as `refund` used in one class. Repeat the
   coefficient audit — can you still see it?
2. **Two-word trigger.** Poison with a bigram trigger and rerun Step 3. Does the coefficient audit still find
   it, and what do you need to change?
3. **Data contract.** Write the five clauses you would require from a fine-tuning data supplier, given this
   demonstration.
4. **Quantify the cost.** Assume 2.5 % poisoning. Compute how many poisoned rows are needed for the effect at
   1 %, and what a 100 % human review of a 10 000-row set would cost.

## Reflection questions

1. Why does a coefficient audit work here but not on a large transformer?
2. Sanitisation removed the exploit. Why is it not a fix?
3. Who is accountable when a fine-tune is poisoned: the data supplier, the trainer, or the deployer?

## References

1. Gu, T., Dolan-Gavitt, B. and Garg, S., "BadNets", 2017.
2. Kurita, K., Michel, P. and Neubig, G., "Weight Poisoning Attacks on Pre-trained Models", ACL 2020.
3. OWASP, *Top 10 for LLM Applications*, 2025 — LLM05, LLM04.
4. NIST, *AI 600-1*, Generative AI Profile, 2024 — data provenance.